In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset
import pandas as pd
import torch
import librosa

device = "cuda" if torch.cuda.is_available() else "cpu"

stt_model_path = "../stt/models/whisper_small_mansi_v2"

processor = WhisperProcessor.from_pretrained(stt_model_path, language="hungarian", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(stt_model_path).to(device)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.forced_decoder_ids = None
model.generation_config._from_model_config = True

In [ ]:
from f5_tts.api import F5TTS

model_name = "mansi_v5_Rus"

model_path = f"../tts/models/{model_name}/model.pt"
vocab_path = f"../tts/models/{model_name}/vocab.txt"

model_f5tts = F5TTS(ckpt_file=model_path, vocab_file=vocab_path, device=device)
pred_path = f'pred_{model_name}'

voices = { 
0: {'ref_audio_path' : '../data/tts_dataset/train/wavs/0a7de388-67fb-465b-92b6-99bf0422d118.wav', 'ref_text' : 'Туня нэ̄мхо̄т ат суйтыгла̄лы, ат на̄ӈкила̄лы, тав сэ̄ръёр.'},
1: {'ref_audio_path' : '../data/stt_dataset/train/wavs/ff387601-10fd-46af-98fa-2346dddfeb5a-1.wav', 'ref_text' : 'Аквматнакт таквсы э̄ти тав ояс.'},
2: {'ref_audio_path' : '../data/stt_dataset/train/wavs/0a0ef1d2-5954-4244-b03a-b91b5ec9f22d-1.wav', 'ref_text' : 'На̄е састум на̄йыӈ са̄мныл.'},
3: {'ref_audio_path' : '../data/stt_dataset/train/wavs/fff537a5-71d9-4caf-a9f6-a73188a67c64-1.wav', 'ref_text' : 'Ка̄тра пун яктуӈкв'},
4: {'ref_audio_path' : '../data/stt_dataset/train/wavs/ff92b4ac-8786-4e66-b4b5-9a9a4aef9ccf-1.wav', 'ref_text' : 'Яныг Ва̄тахум ла̄ви: А̄ти, ам ат ойтхатэ̄гум!'},
}

In [ ]:
# from pathlib import Path
# import random
# import os

# # dataset = pd.read_csv('seq2seq_dataset/train.csv', delimiter="\t")
# dataset = pd.DataFrame(columns=['whisper_out', 'label'])


# pred_path = '../data/synthetic_dataset/wavs'
# os.makedirs(pred_path)

# corpus = pd.read_csv('../data/synthetic_dataset/corpus_mansi.csv')

# for i in range(len(corpus)):
#     j = random.randint(0, 4)
#     sample = corpus.iloc[i]
#     file_path = Path(f'{pred_path}/{i}.wav')
#     if not file_path.exists():
#         print(f'{pred_path}/{i}.wav')
#         model_f5tts.infer(
#             ref_file=voices[j]['ref_audio_path'],
#                 ref_text=voices[j]['ref_text'],
#                 gen_text=sample['text'],
#                 file_wave=f'{pred_path}/{i}.wav',
#                 speed=0.75
#             )

#         pred_file, sr = librosa.load(f'{pred_path}/{i}.wav', sr=16000)
#         test_text = sample['text']

#         inputs = processor(pred_file, sampling_rate=16000, return_tensors="pt").to(device)
#         pred_ids = model.generate(inputs.input_features, max_length=225)
#         pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]
#         new_row = pd.DataFrame([{'whisper_out': pred_text, 'label': test_text}])
#         dataset = pd.concat([dataset, new_row], ignore_index=True)
#         dataset.to_csv('seq2seq_dataset/train.csv', index=False, sep="\t")
    

In [ ]:
stt_path = "../data/stt_dataset/"
tts_path = "../data/tts_dataset/"
bonus_path = "../data/bonus_dataset/"

stt_test_dataset = load_dataset('csv', data_files=stt_path+'test/metadata.csv', delimiter='\t', split='train')
stt_val_dataset = load_dataset('csv', data_files=stt_path+'val/metadata.csv', delimiter='\t', split='train')

tts_test_dataset = load_dataset('csv', data_files=tts_path+'test/metadata.csv', delimiter='\t', split='train')
tts_val_dataset = load_dataset('csv', data_files=tts_path+'val/metadata.csv', delimiter='\t', split='train')

bonus_test_dataset = load_dataset('csv', data_files=bonus_path+'test/metadata.csv', delimiter='\t', split='train')
bonus_val_dataset = load_dataset('csv', data_files=bonus_path+'val/metadata.csv', delimiter='\t', split='train')

In [ ]:
dataset = pd.DataFrame(columns=['whisper_out', 'label'])

for sample in stt_test_dataset:
    pred_file, sr = librosa.load(f'{stt_path}test/wavs/{sample['audio_file']}', sr=16000)
    test_text = sample['text']

    inputs = processor(pred_file, sampling_rate=16000, return_tensors="pt").to(device)
    pred_ids = model.generate(inputs.input_features, max_length=225)
    pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]
    new_row = pd.DataFrame([{'whisper_out': pred_text, 'label': test_text}])
    dataset = pd.concat([dataset, new_row], ignore_index=True)
    # dataset.to_csv('seq2seq_dataset/test.csv', index=False, sep="\t")

for sample in tts_test_dataset:
    pred_file, sr = librosa.load(f'{tts_path}test/wavs/{sample['audio_file']}', sr=16000)
    test_text = sample['text']

    inputs = processor(pred_file, sampling_rate=16000, return_tensors="pt").to(device)
    pred_ids = model.generate(inputs.input_features, max_length=225)
    pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]
    new_row = pd.DataFrame([{'whisper_out': pred_text, 'label': test_text}])
    dataset = pd.concat([dataset, new_row], ignore_index=True)
    # dataset.to_csv('seq2seq_dataset/test.csv', index=False, sep="\t")

for sample in bonus_test_dataset:
    pred_file, sr = librosa.load(f'{bonus_path}test/wavs/{sample['audio_file']}', sr=16000)
    test_text = sample['text']

    inputs = processor(pred_file, sampling_rate=16000, return_tensors="pt").to(device)
    pred_ids = model.generate(inputs.input_features, max_length=225)
    pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]
    new_row = pd.DataFrame([{'whisper_out': pred_text, 'label': test_text}])
    dataset = pd.concat([dataset, new_row], ignore_index=True)
    dataset.to_csv('seq2seq_dataset/test.csv', index=False, sep="\t")


In [ ]:
dataset = pd.DataFrame(columns=['whisper_out', 'label'])

for sample in stt_val_dataset:
    pred_file, sr = librosa.load(f'{stt_path}val/wavs/{sample['audio_file']}', sr=16000)
    test_text = sample['text']

    inputs = processor(pred_file, sampling_rate=16000, return_tensors="pt").to(device)
    pred_ids = model.generate(inputs.input_features, max_length=225)
    pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]
    new_row = pd.DataFrame([{'whisper_out': pred_text, 'label': test_text}])
    dataset = pd.concat([dataset, new_row], ignore_index=True)
    # dataset.to_csv('seq2seq_dataset/val.csv', index=False, sep="\t")

for sample in tts_val_dataset:
    pred_file, sr = librosa.load(f'{tts_path}val/wavs/{sample['audio_file']}', sr=16000)
    test_text = sample['text']

    inputs = processor(pred_file, sampling_rate=16000, return_tensors="pt").to(device)
    pred_ids = model.generate(inputs.input_features, max_length=225)
    pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]
    new_row = pd.DataFrame([{'whisper_out': pred_text, 'label': test_text}])
    dataset = pd.concat([dataset, new_row], ignore_index=True)
    # dataset.to_csv('seq2seq_dataset/val.csv', index=False, sep="\t")

for sample in bonus_val_dataset:
    pred_file, sr = librosa.load(f'{bonus_path}val/wavs/{sample['audio_file']}', sr=16000)
    test_text = sample['text']

    inputs = processor(pred_file, sampling_rate=16000, return_tensors="pt").to(device)
    pred_ids = model.generate(inputs.input_features, max_length=225)
    pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]
    new_row = pd.DataFrame([{'whisper_out': pred_text, 'label': test_text}])
    dataset = pd.concat([dataset, new_row], ignore_index=True)
    dataset.to_csv('seq2seq_dataset/val.csv', index=False, sep="\t")
